# Conditional Autoencoder for Asset Pricing - Part 1: The Data

## Notebook Description

This notebook is the **first part of a project on Conditional Autoencoders for Asset Pricing**. Its primary focus is on **data preparation and feature engineering** required before training the model. Specifically, it covers:

* **Data Loading**: Importing price series and metadata for a wide cross-section of stocks.
* **Return Construction**: Converting raw prices into weekly and monthly returns.
* **Factor Engineering**: Creating predictor variables commonly used in empirical asset pricing, including:

  * Short-term reversal (1-month cumulative return)
  * Momentum (11-month cumulative return, lagged 1 month)
  * Momentum change and additional price-trend signals
* **Rolling Regressions**: Using `statsmodels` to compute time-varying betas and risk factor exposures.
* **Data Storage**: Organizing processed features and results in a structured format for subsequent modeling.

This workflow establishes the **input dataset for conditional autoencoders**, ensuring that returns and characteristics are consistently aligned across time and tickers. The resulting dataset will be used in later notebooks for training and evaluating machine learning models for asset pricing.

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
sns.set_style('whitegrid')

In [4]:
idx = pd.IndexSlice

**NOTE**. The instruction

```python
idx = pd.IndexSlice
```

creates a convenient shorthand for working with **multi-level indexes** (known as *MultiIndex*) in a `pandas` DataFrame.

In this project, the dataset of stock prices is organized using a two-level index:

1. the **ticker** symbol, identifying each traded asset, and
2. the **date**, corresponding to each observation in time.

When a DataFrame has such a hierarchical structure, it is often necessary to **select subsets of data** that depend on both levels of the index — for example, retrieving all data for a particular ticker over a specific date range, or selecting a given period across all securities.

The object `pd.IndexSlice` provides a more readable and flexible way to perform these complex selections using the `.loc[]` indexer. By assigning it to the variable `idx`, the code gains a compact alias that simplifies subsequent commands.

For instance, later in the script we find the expression:

```python
prices_adj.sort_index().loc[idx[:, '1990':'2019'], :]
```

Here, `idx[:, '1990':'2019']` uses the alias defined earlier to specify a *slice* of the MultiIndex.

* The colon (`:`) before the comma means “include all tickers”;
* The range `'1990':'2019'` selects all dates between 1990 and 2019.

Thus, this line extracts every stock’s price data between those years, using the concise syntax made possible by the earlier definition of `idx = pd.IndexSlice`.

In summary, this instruction does not perform any data operation by itself; rather, it defines a convenient reference that improves code readability and facilitates advanced subsetting in DataFrames with hierarchical indexes.


In [5]:
results_path = Path('c:\/','data','ip_2026', 'asset_pricing')

if not results_path.exists():
    results_path.mkdir(parents=True)
print(results_path)

c:\data\ip_2026\asset_pricing


## Load Data

The second step of the notebook concerns the **loading and inspection of financial data** that were previously downloaded and stored in hierarchical format during the data acquisition phase. This section ensures that the adjusted stock prices are properly imported, structured, and verified before being used in the subsequent stages of model training.

The dataset is retrieved from the local storage path `results_path`, which was defined earlier in the workflow as the main directory for processed financial data. Specifically, the program loads the adjusted daily price series for all available stocks from the HDF5 file named `data_reloaded_test.h5`. 

### Prices

In [6]:
prices = pd.read_hdf(results_path / 'data_ip_2026.h5', 'stocks/prices/adjusted')

This command uses the `pandas` function `read_hdf()` to access the hierarchical data file and extract the dataset stored under the key `'stocks/prices/adjusted'`.
The HDF5 format is particularly suitable for this task because it allows for efficient reading and writing of large-scale datasets while maintaining the hierarchical organization of multiple data components—such as prices, metadata, and returns—within a single file.

After the data are loaded into the `prices` variable (a `pandas.DataFrame`), several exploratory operations are executed to verify its structure and integrity.
First, the command:

In [8]:
prices.head()

Price                  close       high        low       open      volume
ticker date                                                              
A      1999-11-18  26.300018  29.886386  23.909107  27.196609  62546380.0
       1999-11-19  24.133266  25.702302  23.797044  25.664942  15234146.0
       1999-11-22  26.300018  26.300018  23.946465  24.693624   6577870.0
       1999-11-23  23.909115  26.075879  23.909115  25.403435   5975611.0
       1999-11-24  24.544199  25.067211  23.909112  23.983829   4843231.0

As we can see this command displays the first few rows of the dataset, offering a preliminary view of the data structure, which typically includes variables such as `open`, `high`, `low`, `close`, and `volume`, organized in a multi-index format where the two index levels correspond to the **ticker symbol** and the **date**.

The following instruction, produces a concise summary of the DataFrame, including the number of entries, column data types, and counts of non-missing values. This step is essential to ensure that the dataset does not contain structural anomalies—such as missing columns, duplicate index entries, or incorrect datatypes—that could affect the stability of the subsequent econometric or machine-learning procedures.

In [10]:
prices.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 14064325 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZYME', Timestamp('2019-12-31 00:00:00'))
Data columns (total 5 columns):
 #   Column  Non-Null Count     Dtype  
---  ------  --------------     -----  
 0   close   14064325 non-null  float64
 1   high    14064325 non-null  float64
 2   low     14064325 non-null  float64
 3   open    14064325 non-null  float64
 4   volume  14064325 non-null  float64
dtypes: float64(5)
memory usage: 591.1+ MB


Finally, the notebook prints key attributes of the index to confirm that the hierarchical structure is correctly preserved:

This verification confirms that the `MultiIndex` is composed of two hierarchical levels:

1. The **ticker**, identifying the individual financial instrument (e.g., “AAPL” for Apple Inc.), and
2. The **date**, representing the trading day associated with each price observation.

This two-dimensional index structure enables convenient slicing and aggregation operations, for instance allowing the user to select all historical data for a given stock or to perform cross-sectional analyses on a specific trading day.
It also provides the necessary data organization for subsequent stages of the project, such as the computation of financial factors, normalization of returns, and conditioning variables used in the autoencoder model.

In [11]:
print(prices.index)
print(prices.index.names)

MultiIndex([(   'A', '1999-11-18'),
            (   'A', '1999-11-19'),
            (   'A', '1999-11-22'),
            (   'A', '1999-11-23'),
            (   'A', '1999-11-24'),
            (   'A', '1999-11-26'),
            (   'A', '1999-11-29'),
            (   'A', '1999-11-30'),
            (   'A', '1999-12-01'),
            (   'A', '1999-12-02'),
            ...
            ('ZYME', '2019-12-17'),
            ('ZYME', '2019-12-18'),
            ('ZYME', '2019-12-19'),
            ('ZYME', '2019-12-20'),
            ('ZYME', '2019-12-23'),
            ('ZYME', '2019-12-24'),
            ('ZYME', '2019-12-26'),
            ('ZYME', '2019-12-27'),
            ('ZYME', '2019-12-30'),
            ('ZYME', '2019-12-31')],
           names=['ticker', 'date'], length=14064325)
['ticker', 'date']


In summary, this section establishes the analytical foundation of the entire modeling process. By loading the adjusted historical prices into a structured `pandas` DataFrame and validating their internal consistency, it guarantees that all subsequent operations—ranging from feature construction to neural-network training—are based on a coherent and reliable dataset.

### Metadata

This section focuses on loading and inspecting the **metadata** associated with each traded security.
While price data provide the temporal evolution of market values, metadata supply the **cross-sectional attributes** of each stock — such as its name, industry, sector, country, and market capitalization.
These descriptive variables are essential for many econometric and machine-learning applications, as they allow models to condition on firm-level characteristics or to group assets by category.

The metadata were previously downloaded from Yahoo Finance and saved in the same HDF5 file that stores the adjusted price series. The corresponding dataset is now retrieved with the following instruction:

In [12]:
metadata = pd.read_hdf(results_path / 'data_ip_2026.h5', 'stocks/info').rename(columns=str.lower)

Here, the `pandas.read_hdf()` function accesses the hierarchical data file located in the `results_path` directory and reads the content stored under the key `'stocks/info'`. The variable `metadata` therefore becomes a `pandas.DataFrame` where each row corresponds to a distinct stock (identified by its ticker symbol) and each column represents a descriptive attribute obtained from Yahoo Finance’s API, such as:

* company name and full trading symbol,
* market and country of listing,
* sector and industry classification,
* market capitalization, dividend yield, and beta coefficient,
* financial ratios and other fundamental indicators.

Once loaded, the notebook typically executes exploratory commands such as:

In [13]:
metadata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7401 entries, A to ZYME
Columns: 210 entries, address1 to exchangetransferdate
dtypes: float64(154), object(56)
memory usage: 11.9+ MB


In [14]:
metadata.head()

,address1,city,state,zip,country,phone,website,industry,industrykey,industrydisp,...,trailingthreemonthnavreturns,netassets,netexpenseratio,newlistingdate,industrysymbol,openinterest,prevticker,tickerchangedate,prevexchange,exchangetransferdate
A,5301 Stevens Creek Boulevard,Santa Clara,CA,95051,United States,800 227 9770,https://www.agilent.com,Diagnostics & Research,diagnostics-research,Diagnostics & Research,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AA,201 Isabella Street,Pittsburgh,PA,15212-5858,United States,412-315-2900,https://www.alcoa.com,Aluminum,aluminum,Aluminum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AACB,3 Columbus Circle,New York,NY,10019,United States,212-309-7668,https://www.artiuscapital.com/acquisition,Shell Companies,shell-companies,Shell Companies,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AACBR,3 Columbus Circle,New York,NY,10019,United States,212-309-7668,https://www.artiuscapital.com/acquisition,,,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AACBU,3 Columbus Circle,New York,NY,10019,United States,212-309-7668,https://www.artiuscapital.com/acquisition,Shell Companies,shell-companies,Shell Companies,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The first command (`meta.head()`) displays the top rows of the dataset, offering a quick overview of the available variables and their typical structure.
The second (`meta.info(show_counts=True)`) provides a summary of the DataFrame, including data types, the number of non-null entries per column, and memory usage.
This allows the user to assess the completeness and quality of the metadata — for example, detecting missing sector classifications or incomplete fundamental information for certain tickers.

The metadata play a crucial role in the next phases of analysis.
They can be used to define **conditioning variables** for the autoencoder model, enabling the network to differentiate between stocks based on structural or economic characteristics.
They also allow for **cross-sectional filtering**, such as restricting the analysis to a specific sector (e.g., technology or energy) or to firms meeting certain criteria (e.g., market capitalization above a threshold).

In summary, this paragraph ensures that the descriptive information linked to each stock is properly loaded, inspected, and validated before being merged with the time-series data.
Together, the metadata and price series form a coherent dataset that integrates both the temporal and structural dimensions of the financial market — a necessary foundation for building the conditional autoencoder used in the later stages of the project.

### Select tickers with metadata

After both the price data and the metadata have been successfully loaded, the next essential step is to ensure that the two datasets are **perfectly aligned** — that is, that they refer to the same set of tickers.
Because the Yahoo Finance download process can occasionally fail for certain securities (for instance, due to delistings, missing records, or inconsistent identifiers), the list of tickers available in the price dataset may not coincide exactly with those present in the metadata table.
To guarantee consistency across all analyses, the notebook performs a filtering operation that retains only the securities for which both types of information are available.

First of all, we define a variable `sectors`.This command identifies the industry sectors that are sufficiently represented in the dataset — in this case, those containing more than 50 companies.

In [15]:
sectors = (metadata.sector.value_counts() > 50).index

1. **`metadata.sector`**
   This selects the column `sector` from the `metadata` DataFrame.
   It is a `pandas.Series` containing the sector label (e.g., “Technology,” “Financials,” “Healthcare”) for each stock.

2. **`value_counts()`**
   This method counts how many times each unique sector appears in the column.
   It returns a `Series` indexed by sector names, with the corresponding counts as values, sorted in descending order.
   For example:

   ```
   Technology      320
   Financials      210
   Healthcare      145
   ...
   ```

   **By default, it excludes missing values (`NaN`)**.

3. **`> 50`**
   This comparison applies a Boolean filter to the count values, testing whether each sector has more than 50 stocks.
   The result is a Boolean Series:

   ```
   Technology      True
   Financials      True
   Healthcare      True
   ...
   Utilities       False
   ```

   The threshold of 50 acts as a **minimum sample-size condition**, ensuring that only sectors with a sufficiently large number of observations are retained for reliable analysis.

The resulting variable `sectors` will be used to **filter the dataset** to include only the well-represented sectors.
For example:

```python
metadata_filtered = metadata[metadata.sector.isin(sectors)]
```

This operation ensures that subsequent analyses — such as factor computation, sector-level regressions, or machine-learning model training — are based on groups of stocks large enough to produce stable and meaningful results.In short, this command identifies the **well-populated industry sectors** in the dataset — those with more than 50 companies. This filtering step ensures that subsequent analyses focus on sectors with sufficient representation, improving the statistical reliability and stability of any models or empirical results that depend on sector-level information.

In [16]:
tickers_with_errors = []

In [17]:
tickers_with_metadata = metadata[metadata.sector.isin(sectors) & 
                                 metadata.marketcap.notnull() &
                                 metadata.sharesoutstanding.notnull() & 
                                (metadata.sharesoutstanding > 0)].index.drop(tickers_with_errors)

The previous command defines the subset of **valid stock tickers** that meet a set of quality and completeness criteria in the `metadata` dataset. It filters out companies with missing or invalid information, ensuring that only securities with reliable descriptive attributes are retained for further analysis. Let’s break down the logic step by step.

---

#### 1. `metadata[...]`

The expression inside the square brackets (`[...]`) is a Boolean mask applied to the `metadata` DataFrame.
Only the rows for which all conditions inside the brackets evaluate to `True` will be kept.
Each condition corresponds to a different requirement for the quality and completeness of the data.

---

#### 2. `metadata.sector.isin(sectors)`

This condition keeps only those stocks whose `sector` belongs to the list of well-represented sectors previously identified and stored in the variable `sectors` (for example, sectors with more than 50 companies).
In other words, it removes all firms that belong to small or poorly populated sectors.

---

#### 3. `metadata.marketcap.notnull()`

This condition ensures that the variable `marketcap` (market capitalization) is **not missing**.
Market capitalization is a key variable in asset-pricing studies, as it measures the total market value of a company’s outstanding shares and is often used to compute size-related factors (e.g., the SMB factor in Fama–French models).
Any observation lacking this value would be unreliable for downstream financial computations.

---

#### 4. `metadata.sharesoutstanding.notnull()`

Similarly, this filter keeps only those companies for which the number of outstanding shares (`sharesoutstanding`) is available.
This information is essential to verify or reconstruct market capitalization and to perform any analysis involving shares volume or float-adjusted measures.

---

#### 5. `(metadata.sharesoutstanding > 0)`

This condition guarantees that the number of shares outstanding is strictly positive.
Occasionally, data sources can contain placeholder or erroneous zero values, which would lead to division-by-zero errors or meaningless ratios (for example, when computing price-per-share metrics).
By enforcing this constraint, the dataset retains only economically valid entries.

---

#### 6. Combining conditions with `&`

The ampersand operator `&` combines all four Boolean conditions using a logical “AND”.
Thus, a stock will be selected **only if all criteria are simultaneously satisfied** — that is, it belongs to a valid sector, has non-missing market capitalization, has non-missing shares outstanding, and that number is greater than zero.

The result of this combined condition is a **filtered DataFrame** containing only those rows that meet all these requirements.

---

#### 7. `.index`

Once the filtered DataFrame is created, the `.index` attribute extracts the list of index labels, which in this dataset correspond to the **ticker symbols** (e.g., “AAPL”, “MSFT”, “GOOG”).
This produces an index object containing all the valid tickers that passed the filtering process.

---

#### 8. `.drop(tickers_with_errors)`

Finally, the method `.drop(tickers_with_errors)` removes from this list any tickers known to be problematic.
The variable `tickers_with_errors` typically contains a list of symbols that generated download or processing errors during earlier stages (for example, incomplete price histories, API failures, or inconsistent metadata).
Dropping them ensures that the final set `tickers_with_metadata` includes only fully functional, reliable securities.

---

In plain terms, this line of code constructs a **clean and trustworthy subset** of stock identifiers to be used in the modeling pipeline.
It systematically excludes:

* Sectors with insufficient data,
* Firms missing crucial financial variables (market capitalization or shares outstanding), and
* Securities affected by known errors.

The resulting variable `tickers_with_metadata` therefore defines the **effective universe of assets** that will enter the autoencoder-based asset-pricing model.
This careful pre-selection step is critical in empirical finance and machine learning alike, as it prevents missing or corrupted data from introducing bias or instability into subsequent computations and training procedures.

In [18]:
tickers_with_metadata

Index(['A', 'AA', 'AACB', 'AACBU', 'AACG', 'AAL', 'AAME', 'AAMI', 'AAOI',
       'AAON',
       ...
       'ZTO', 'ZTR', 'ZTS', 'ZUMZ', 'ZURA', 'ZVIA', 'ZVRA', 'ZWS', 'ZYBT',
       'ZYME'],
      dtype='object', length=5843)

The following two lines refine and standardize the structure of the metadata DataFrame, preparing it for integration with other datasets (in particular, the price time series).
They perform two distinct but complementary tasks:
(1) selecting only the relevant subset of rows and columns, and
(2) ensuring that the index is explicitly labeled with a meaningful name.

In [20]:
metadata = metadata.loc[tickers_with_metadata, ['sector', 'sharesoutstanding', 'marketcap']]

**Selecting relevant rows and columns.**
This line uses the `.loc[]` indexer from the `pandas` library, which allows **label-based selection** of rows and columns within a DataFrame.

* The **row selection** is defined by `tickers_with_metadata`, a list of ticker symbols that were previously identified as valid and complete (i.e., belonging to well-represented sectors, with non-missing and positive `marketcap` and `sharesoutstanding` values).
  By specifying this list inside `.loc[ ... , ]`, the code filters the DataFrame so that only those companies remain.

* The **column selection** is defined by the list `['sector', 'sharesoutstanding', 'marketcap']`.
  This restricts the DataFrame to include only the variables that are essential for the next analytical steps:

  * `sector`: a categorical descriptor used to group or condition stocks by industry;
  * `sharesoutstanding`: the total number of shares issued by each company;
  * `marketcap`: the market capitalization (share price multiplied by shares outstanding), representing the firm’s size.

As a result, the new `metadata` DataFrame contains one row per valid ticker and only these three columns, ensuring a compact and consistent structure. This type of targeted filtering is a good practice in data preprocessing: it eliminates unnecessary variables, reduces memory usage, and minimizes the risk of inconsistencies when merging with other datasets (for example, price data indexed by the same tickers).

In [21]:
metadata.index.name = 'ticker'

**Naming the index explicitly.**
This second instruction assigns the name `'ticker'` to the DataFrame’s index.
In `pandas`, the index serves as the unique identifier for each row.
By explicitly naming it, the dataset becomes **self-descriptive** and **easier to merge** with other DataFrames that use the same indexing convention.

For example, when later joining this metadata with price or returns data (which also use ticker symbols as part of a `MultiIndex`), the named index ensures that the join operation is unambiguous and robust.

In [22]:
metadata.head()

,sector,sharesoutstanding,marketcap
ticker,,,
A,Healthcare,282602317.0,3.148005e+10
AA,Basic Materials,263839742.0,1.489375e+10
AACB,Financial Services,22175000.0,2.861595e+08
AACBU,Financial Services,25175000.0,2.693725e+08
AACG,Consumer Defensive,31772461.0,3.272563e+07


In [23]:
print(len(set(tickers_with_metadata)))
#print(set(tickers_with_metadata))

5843


In [24]:
tickers_in_prices = prices.index.get_level_values(0).unique()
print(len(set(tickers_in_prices)))
#print(set(tickers_in_prices))

3319


In [25]:
print(len(set(tickers_with_metadata)-set(tickers_in_prices)))
#print(set(tickers_with_metadata)-set(tickers_in_prices))

2612


The following two lines perform a **final consistency check and synchronization** between the metadata and the price dataset.
Their purpose is to ensure that both datasets refer to the exact same set of securities — that is, only to tickers for which both descriptive attributes (metadata) and historical price series are available. This alignment step is essential before proceeding to compute returns, factors, or before feeding the data into the conditional autoencoder.

---

**Identifying valid tickers present in both datasets**

In [27]:
valid_tickers = tickers_with_metadata.intersection(prices.index.get_level_values(0).unique())

This line creates a list of ticker symbols that exist simultaneously in both the **metadata** and the **prices** datasets.

Let’s break it down step by step:

* **`tickers_with_metadata`**
  This is the list of tickers that passed all quality filters on the metadata side — for example, belonging to valid sectors and having complete information on market capitalization and shares outstanding.

* **`prices.index.get_level_values(0).unique()`**
  The price dataset, `prices`, is organized as a `pandas` **MultiIndex DataFrame**, where the first level of the index (`level 0`) corresponds to the **ticker** and the second level to the **date**.
  The method `get_level_values(0)` retrieves all ticker labels from that first level.
  The call to `.unique()` then extracts the distinct ticker symbols actually present in the price data.

* **`.intersection(...)`**
  The method `intersection()` computes the set of tickers that appear in both collections:
  those available in the metadata and those with valid price histories.
  The resulting object, `valid_tickers`, therefore contains the **common universe of securities** for which the two data sources are consistent.

This intersection ensures that only stocks with both fundamental descriptors and price time series are retained, eliminating any mismatched or incomplete entries.

---

**Filtering the price dataset accordingly**

In [28]:
prices = prices.loc[idx[valid_tickers, :], :]

This line filters the `prices` DataFrame so that it contains only the rows corresponding to the `valid_tickers` identified above.

* **`idx`**
  As defined earlier (`idx = pd.IndexSlice`), this is a convenient shorthand used to simplify slicing operations on MultiIndex DataFrames.

* **`idx[valid_tickers, :]`**
  The first element (`valid_tickers`) selects all rows corresponding to the tickers of interest in the first level of the index,
  while the colon (`:`) after the comma selects **all dates** associated with each ticker.

* **`.loc[ ... , :]`**
  The `.loc[]` indexer applies this two-dimensional selection to the `prices` DataFrame, keeping all variables (columns) for the chosen rows.

As a result, the command retains in `prices` only those time-series observations that belong to the tickers present in both datasets.

---

Together, these two lines finalize the **data alignment** between the fundamental (metadata) and market (prices) datasets.
The logic ensures that:

1. Every stock in the price dataset also appears in the metadata dataset, and vice versa;
2. No stock with incomplete or inconsistent data is included in subsequent computations.

This synchronization step is crucial in what follows. By guaranteeing that every observation has both fundamental descriptors and market information, the data become **internally coherent**, preventing errors in later stages — for example, when joining tables, computing returns, or conditioning the autoencoder on firm-level variables. In essence, this operation defines the **final working universe** of securities for the project: a clean, consistent, and fully cross-referenced set of assets suitable for robust empirical analysis.

In [29]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 13930760 entries, ('A', Timestamp('1999-11-18 00:00:00')) to ('ZYME', Timestamp('2019-12-31 00:00:00'))
Data columns (total 5 columns):
 #   Column  Dtype  
---  ------  -----  
 0   close   float64
 1   high    float64
 2   low     float64
 3   open    float64
 4   volume  float64
dtypes: float64(5)
memory usage: 585.5+ MB


In [30]:
prices.head()

Price                  close       high        low       open      volume
ticker date                                                              
A      1999-11-18  26.300018  29.886386  23.909107  27.196609  62546380.0
       1999-11-19  24.133266  25.702302  23.797044  25.664942  15234146.0
       1999-11-22  26.300018  26.300018  23.946465  24.693624   6577870.0
       1999-11-23  23.909115  26.075879  23.909115  25.403435   5975611.0
       1999-11-24  24.544199  25.067211  23.909112  23.983829   4843231.0

In [31]:
prices.close.unstack('ticker').head()

ticker,A,AA,AAL,AAMI,AAOI,AAON,AAP,AAPL,AAT,AB,...,ZM,ZS,ZTEK,ZTO,ZTR,ZTS,ZUMZ,ZVRA,ZWS,ZYME
date,,,,,,,,,,,,,,,,,,,,,
1999-11-18,26.300018,53.288662,NaN,NaN,NaN,0.625539,NaN,0.671358,NaN,4.833004,...,NaN,NaN,NaN,NaN,1.804794,NaN,NaN,NaN,NaN,NaN
1999-11-19,24.133266,54.299122,NaN,NaN,NaN,0.625539,NaN,0.692426,NaN,4.814270,...,NaN,NaN,NaN,NaN,1.804794,NaN,NaN,NaN,NaN,NaN
1999-11-22,26.300018,54.990494,NaN,NaN,NaN,0.625539,NaN,0.678850,NaN,4.692507,...,NaN,NaN,NaN,NaN,1.788822,NaN,NaN,NaN,NaN,NaN
1999-11-23,23.909115,55.415932,NaN,NaN,NaN,0.613509,NaN,0.695235,NaN,4.655042,...,NaN,NaN,NaN,NaN,1.772851,NaN,NaN,NaN,NaN,NaN
1999-11-24,24.544199,54.684696,NaN,NaN,NaN,0.619524,NaN,0.709280,NaN,4.617579,...,NaN,NaN,NaN,NaN,1.788822,NaN,NaN,NaN,NaN,NaN


In [32]:
metadata.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5843 entries, A to ZYME
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sector             5843 non-null   object 
 1   sharesoutstanding  5843 non-null   float64
 2   marketcap          5843 non-null   float64
dtypes: float64(2), object(1)
memory usage: 182.6+ KB


In [33]:
close = prices.close.unstack('ticker').sort_index()
close.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7559 entries, 1990-01-02 to 2019-12-31
Columns: 3231 entries, A to ZYME
dtypes: float64(3231)
memory usage: 186.4 MB


In [34]:
volume = prices.volume.unstack('ticker').sort_index()
volume.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 7559 entries, 1990-01-02 to 2019-12-31
Columns: 3231 entries, A to ZYME
dtypes: float64(3231)
memory usage: 186.4 MB


### Create weekly returns

This section transforms the raw daily price data into **weekly returns**, a more stable and analytically convenient representation of each asset’s performance over time.
Daily observations, although precise, tend to be noisy due to short-term volatility, market microstructure effects, and day-to-day fluctuations that are often irrelevant to long-horizon modeling.
Aggregating them to the weekly level reduces this noise, lowers data dimensionality, and provides a better balance between temporal resolution and statistical robustness.

The procedure begins from the `prices` DataFrame, which contains adjusted closing prices for each stock organized in a multi-index format: the first level corresponds to the ticker symbol, and the second to the date.
To compute returns, the adjusted prices are first **unstacked** so that each column represents a different ticker and each row corresponds to a specific trading date.
This structure makes it easy to apply time-series operations simultaneously across all securities.

Next, the daily prices are **resampled** to weekly frequency using `pandas`’ built-in time-series functions, typically with the command `resample('W')`.
The resampling process aggregates the last available price in each week, which represents the closing price at the end of that period.
Weekly returns are then calculated as the **percentage change** between consecutive weekly closing prices using the `pct_change()` method:

In [35]:
print(prices.close.unstack('ticker').resample('W-FRI').last().sort_index().head())

ticker       A         AA  AAL  AAMI  AAOI  AAON  AAP      AAPL  AAT  \
date                                                                   
1990-01-05 NaN  12.880844  NaN   NaN   NaN   NaN  NaN  0.264205  NaN   
1990-01-12 NaN  12.521278  NaN   NaN   NaN   NaN  NaN  0.241459  NaN   
1990-01-19 NaN  11.125326  NaN   NaN   NaN   NaN  NaN  0.239709  NaN   
1990-01-26 NaN  10.575410  NaN   NaN   NaN   NaN  NaN  0.229211  NaN   
1990-02-02 NaN  10.801896  NaN   NaN   NaN   NaN  NaN  0.239709  NaN   

ticker            AB  ...  ZM  ZS  ZTEK  ZTO       ZTR  ZTS  ZUMZ  ZVRA  ZWS  \
date                  ...                                                      
1990-01-05  0.271088  ... NaN NaN   NaN  NaN  0.970473  NaN   NaN   NaN  NaN   
1990-01-12  0.264735  ... NaN NaN   NaN  NaN  0.958188  NaN   NaN   NaN  NaN   
1990-01-19  0.268971  ... NaN NaN   NaN  NaN  0.958188  NaN   NaN   NaN  NaN   
1990-01-26  0.262617  ... NaN NaN   NaN  NaN  0.958188  NaN   NaN   NaN  NaN   
1990-02-02  0.2

In [36]:
# Compute weekly returns from daily closing prices

# 1. Select the 'close' column from the prices DataFrame.
#    Assumes 'prices' is a multi-index DataFrame with 'ticker' as one of the index levels.
returns = (prices.close
           # 2. Pivot the data so that each column corresponds to a ticker
           #    and rows correspond to dates. This makes tickers wide-form.
           .unstack('ticker')
           # 3. Resample the data to a weekly frequency, taking the last
           #    available observation on each Friday ('W-FRI').
           #    This ensures consistent weekly time steps.
           .resample('W-FRI').last()
           # 4. Sort the index to make sure dates are in chronological order.
           .sort_index()
           # 5. Compute percentage change from one week to the next.
           #    This gives weekly returns for each ticker.
           .pct_change()
           # 6. Drop the first row because pct_change() introduces a NaN
           #    in the first observation (no previous data to compare).
           .iloc[1:]
          )

# Display information about the resulting DataFrame
# This shows the number of entries, index type, columns, and memory usage.
returns.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1565 entries, 1990-01-12 to 2020-01-03
Freq: W-FRI
Columns: 3231 entries, A to ZYME
dtypes: float64(3231)
memory usage: 38.6 MB


In this typical pattern:

* `unstack('ticker')` reshapes the data so that each ticker becomes a column;
* `resample('W').last()` selects the last price observed each week;
* `pct_change()` computes the week-to-week return for every stock; and
* `stack('ticker')` restores the multi-index structure with levels `ticker` and `date`.

The resulting object, `returns_w`, is a `pandas` DataFrame containing weekly log-like percentage returns for all valid tickers.
Each entry represents the proportional change in adjusted price from one week’s close to the next.
Because the data are adjusted for dividends and splits, these returns reflect total shareholder performance over time.

Finally, any missing or invalid observations (for instance, due to non-trading weeks or incomplete historical records) are removed or set to `NaN`.
This cleaning step ensures that the return series are consistent and can be safely merged with firm-level metadata or used as input features for machine-learning models such as the conditional autoencoder.

In summary, this part of the notebook produces a coherent panel of weekly returns that serves as the foundation for all subsequent analyses.
By converting high-frequency price data into lower-frequency returns, it captures the essential information on asset performance while mitigating excessive noise and improving the statistical reliability of the model’s training dataset.

In [37]:
dates = returns.index

In [38]:
#sns.distplot(returns.count(1), kde=False);

In [39]:
print(results_path)

c:\data\ip_2026\asset_pricing


In [40]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    store.put('close', close)
    store.put('volume', volume)
    store.put('returns', returns)
    store.put('metadata', metadata)

In [41]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

## Factor Engineering

This section focuses on the construction of **explanatory variables**, or *factors*, that will be used to condition and interpret the behavior of asset returns in subsequent stages of the analysis.
In the context of empirical asset pricing, *factors* are variables that capture systematic sources of risk or firm characteristics that help explain cross-sectional differences in expected returns.
Typical examples include measures of size, value, profitability, momentum, and volatility.
The process of factor engineering therefore transforms raw financial and market data into meaningful economic signals that serve as model inputs for the conditional autoencoder.

---

#### 1. Purpose of factor engineering

The goal of this stage is to create a structured set of firm-level variables that describe each stock’s economic and financial profile at a given point in time.
These variables will later act as **conditioning features** for the machine-learning model, enabling it to learn how return dynamics depend on observable firm characteristics.

In traditional econometrics, such factors would correspond to the well-known Fama–French characteristics (e.g., market capitalization, book-to-market ratio, operating profitability, etc.).
Here, the same conceptual foundation is adopted, but the implementation is generalized and automated within the Python environment to support a large cross-section of assets.

---

#### 2. Inputs and data sources

The factors are derived from two main sources:

1. **Market data**, particularly adjusted prices, which are used to compute returns, momentum indicators, and measures of past volatility.
2. **Fundamental metadata**, such as `marketcap` and `sharesoutstanding`, obtained from Yahoo Finance, which provide size-related information and allow normalization of other variables.

As we have already seen, before factor computation, both datasets have been carefully synchronized to ensure consistency across time and tickers — that is, only stocks with valid metadata and complete price histories are included.

### Price Trend

In [42]:
# Number or days in a month
MONTH = 21

#### Short-Term Reversal

**1-month cumulative return**

**Definition:**

Suppose $P_t$ is the **closing price** of a stock at time $t$ (measured in trading days).
Let $M$ denote the number of trading days in a month, typically $M \approx 21$.

The **1-month cumulative return** at time $t$ is defined as:

$$
r^{(1m)}_t \;=\; \frac{P_t}{P_{t-M}} - 1
$$

**Properties**

* **Cumulative, not average:** It reflects the *total compounded growth* over the past month, not an average per day.
* **Frequency:** If you resample to Fridays, $r^{(1m)}_t$ gives the 1-month return *as of each Friday*.
* **Lag structure:** In empirical asset pricing, researchers often use this (or longer horizon versions like 12-month momentum) as predictors of future returns.

In the following code,

```python
close.pct_change(periods=MONTH)
```

is exactly computing

$$
r^{(1m)}_t = \frac{P_t}{P_{t-M}} - 1
$$

for each stock.

In [43]:
dates[:5]

DatetimeIndex(['1990-01-12', '1990-01-19', '1990-01-26', '1990-02-02',
               '1990-02-09'],
              dtype='datetime64[ns]', name='date', freq='W-FRI')

In [44]:
# Compute 1-month momentum signal for each stock
mom1m = (close
         # 1. Compute percentage change in closing prices over a given period.
         #    'MONTH' is assumed to be defined earlier, e.g. MONTH = 21 (approx. trading days in a month).
         #    So this computes the 1-month price return for each stock.
         .pct_change(periods=MONTH, fill_method=None)
         # 2. Resample the data to weekly frequency, taking the last available
         #    value on Friday ('W-FRI').
         #    This ensures the momentum signal is aligned to weekly observations.
         .resample('W-FRI').last()
         # 3. Convert the DataFrame from wide format (dates × tickers) back to
         #    long format, with a MultiIndex (date, ticker).
         .stack()
         # 4. Wrap the result into a single-column DataFrame named 'mom1m'
         #    for clarity and easy joining with other features.
         .to_frame('mom1m')
        )

# Display information about the resulting DataFrame:
# number of rows, index type, column(s), and memory usage.
mom1m.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2875516 entries, (Timestamp('1990-02-02 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   mom1m   float64
dtypes: float64(1)
memory usage: 33.1+ MB


In [45]:
mom1m.head()

mom1m
date       ticker          
1990-02-02 AA     -0.169580
           AAPL   -0.089701
           AB      0.007812
           ABM     0.041350
           ABT    -0.047394

In [46]:
mom1m.squeeze().to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/mom1m')
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

#### Stock Momentum

**11-month cumulative returns ending 1-month before month end**

This means:

* Look back over the **past 12 months** of data.
* **Skip the most recent 1 month** (to avoid short-term reversals and microstructure effects).
* Compute the **cumulative return** from 12 months ago up to 1 month ago.

This is the classic **momentum measure** in asset pricing (Jegadeesh–Titman 1993).

**Analytical formulation**

Let $P_t$ be the closing price at time $t$ (measured in trading days).
Let $M$ = number of trading days in 1 month (≈ 21).

Then the **11-month cumulative return ending 1 month before $t$** is:

$$
r^{(11m,\,skip1)}_t \;=\; \frac{P_{t-M}}{P_{t-12M}} - 1
$$

1. **End point:**

   * Instead of using today’s price $P_t$, we use the price from **1 month ago**, $P_{t-M}$.

2. **Start point:**

   * Go back 12 months from today, i.e. to $P_{t-12M}$.

3. **Cumulative return:**

   * Compare $P_{t-M}$ with $P_{t-12M}$.
   * This captures the compounded growth over **11 months**, ignoring the most recent month.


**Why skip the last month?**

Empirical studies show that **short-term reversal effects** (last month’s losers tend to bounce back) can contaminate momentum signals. By excluding the most recent month, the measure isolates **medium-term momentum**, which is more predictive of future returns.

In [47]:
mom12m = (close
            .pct_change(periods=11 * MONTH)
            .shift(MONTH)
            .resample('W-FRI')
            .last()
            .stack()
            .to_frame('mom12m'))

In [48]:
#mom12m.info(null_counts=True)
mom12m.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2724572 entries, (Timestamp('1991-01-04 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   mom12m  float64
dtypes: float64(1)
memory usage: 31.4+ MB


In [49]:
mom12m.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/mom12m')
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

#### Momentum Change

**Cumulative return from months t-6 to t-1 minus months t-12 to t-7**

This is a **momentum-change** style variable. Let's unpack it carefully. We split the **past 12 months (t-12 to t-1)** into two consecutive 6-month windows:

1. **Recent past (t-6 to t-1):** the most recent six months (excluding the current month).

   $$
   R_{t-6:t-1} = \frac{P_{t-1}}{P_{t-6}} - 1
   $$

2. **Earlier past (t-12 to t-7):** the six months before that.

   $$
   R_{t-12:t-7} = \frac{P_{t-7}}{P_{t-12}} - 1
   $$

The variable is then defined as:

$$
MC_t = R_{t-6:t-1} - R_{t-12:t-7}
$$

where $MC_t$ stands for **momentum change**.


* **Momentum factor (standard):**
  Normally, researchers look at cumulative returns over the past 12 months (skipping the last month). This captures the idea that *winners keep winning, losers keep losing*.

* **Momentum change modification:**
  Instead of just looking at the whole year, this variable compares **recent momentum** (last 6 months) with **earlier momentum** (the 6 months before that).

  * If recent momentum is **higher** than earlier momentum ($MC_t > 0$):
    → the stock’s performance is **accelerating**.
  * If recent momentum is **lower** than earlier momentum ($MC_t < 0$):
    → the stock’s performance is **decelerating**.

* **Why it matters:**

  * Markets often exhibit **momentum crashes** or **reversals** when trends weaken.
  * By looking at the *change* in momentum, investors can detect whether a trend is **strengthening** or **losing steam**.
  * This may help distinguish between:

    * Stocks with sustainable momentum (trend is still strong), and
    * Stocks where momentum is fading (potential reversal risk).

**Example in practice**

Imagine two stocks, both with a 12-month return of +20%:

* **Stock A:** +15% in the last 6 months, +5% in the earlier 6 months → accelerating (positive momentum change).
* **Stock B:** +5% in the last 6 months, +15% in the earlier 6 months → decelerating (negative momentum change).

Even though both had the same 12-month return, their **momentum profiles** are very different — which could lead to different future performance.

**Summary**

The variable

$$
MC_t = (r_{t-6:t-1}) - (r_{t-12:t-7})
$$

is designed to measure **acceleration vs. deceleration in stock momentum**.
It refines the standard momentum factor by splitting it into two windows, providing a signal on whether the trend is getting stronger or weaker.

In [50]:
# Compute 6-to-1 month momentum change ("chmom") for each stock,
# aligned to weekly (Friday) observations and returned in long format.

chmom = (
    close
    # 1) Compute the cumulative return over the last 6 months for each date.
    #    MONTH should be the number of trading days in ~1 month (commonly 21).
    #    So periods=6*MONTH ≈ 126 trading days.
    #    This yields: R_{t-6:t-1} = P_t / P_{t-6M} - 1   (in daily indexing, M = MONTH)
    .pct_change(periods=6 * MONTH)
    # 2) Subtract the 6-month return measured 6 months earlier.
    #    The .shift(6*MONTH) moves that 6-month return back by ~6 months in time,
    #    i.e., it aligns the "earlier 6-month window" (t-12 to t-7) with the current date index.
    #    Result: [R_{t-6:t-1}] - [R_{t-12:t-7}], i.e., momentum acceleration/deceleration.
    .sub(
        close.pct_change(periods=6 * MONTH).shift(6 * MONTH)
    )
    # 3) Resample to weekly frequency, taking the last available value on each Friday.
    #    This aligns the signal to a weekly panel (reduces daily noise and ensures consistency).
    .resample('W-FRI').last()
    # 4) Convert from wide (dates × tickers) to long format with a MultiIndex (date, ticker).
    .stack()
    # 5) Wrap as a single-column DataFrame named 'chmom' for clear downstream merging.
    .to_frame('chmom')
)

In [51]:
chmom.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2724572 entries, (Timestamp('1991-01-04 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   chmom   float64
dtypes: float64(1)
memory usage: 31.4+ MB


In [52]:
chmom.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/chmom')
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

#### Industry Momentum

Equal-weighted avg. industry 12-month returns

In [53]:
# Compute Equal-Weighted Average Industry 12-Month Returns (indmom)

indmom = (
    close
    # 1) Compute 12-month cumulative returns for each stock.
    #    MONTH ≈ 21 trading days → 12*MONTH ≈ 252 trading days (1 year).
    #    This gives: r_t^(12m) = P_t / P_{t-12M} - 1
    .pct_change(12 * MONTH)
    # 2) Resample to weekly frequency, taking the last observation on each Friday.
    #    Aligns all signals to a common weekly time grid.
    .resample('W-FRI').last()
    # 3) Convert from wide format (dates × tickers) back to long format (date, ticker).
    .stack()
    # 4) Wrap into a DataFrame with column name 'close'
    #    (actually contains 12-month return, not price, but labeled 'close' here).
    .to_frame('close')
    # 5) Join with stock metadata to add each stock's sector classification.
    #    'metadata' is assumed to contain a 'sector' column keyed by ticker.
    .join(metadata[['sector']])
    # 6) Group by (date, sector) → for each week and each sector,
    #    collect all stocks that belong to that sector.
    .groupby(['date', 'sector'])
    # 7) Compute the equal-weighted average 12-month return across all stocks in that sector.
    .close.mean()
    # 8) Convert the result (a Series with MultiIndex) back into a DataFrame with column 'indmom'.
    .to_frame('indmom')
    # 9) Reset index so that 'date' and 'sector' become standard columns instead of index levels.
    .reset_index()
)

**What this computes**

* For each **ticker**, compute the **12-month return**

  $$
  r^{(12m)}_{i,t} = \frac{P_{i,t}}{P_{i,t-12M}} - 1
  $$

* Align these returns to **weekly Fridays**.

* Add **sector labels** from metadata.

* For each **sector $s$** and each **week $t$**, compute the **equal-weighted mean return** across all stocks in that sector:

  $$
  \text{indmom}_{s,t} = \frac{1}{N_{s,t}} \sum_{i \in s} r^{(12m)}_{i,t}
  $$

  where $N_{s,t}$ = number of stocks in sector $s$ at time $t$.


**Interpretation**: `indmom` is a panel DataFrame with columns `[date, sector, indmom]`, giving the **average past-year performance of each industry/sector**, updated weekly.

This factor is often used as a **sector-momentum control** in asset pricing, helping to separate **stock-specific momentum** from **industry-wide trends**.

In [54]:
indmom.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18168 entries, 0 to 18167
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    18168 non-null  datetime64[ns]
 1   sector  18168 non-null  object        
 2   indmom  18168 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 425.9+ KB


In [55]:
indmom.head()

,date,sector,indmom
0,1991-01-04,,-0.003784
1,1991-01-04,Basic Materials,-0.084250
2,1991-01-04,Communication Services,-0.277997
3,1991-01-04,Consumer Cyclical,-0.160619
4,1991-01-04,Consumer Defensive,-0.044348


* **Previous block** (`close.pct_change(...).resample(...).groupby(...).mean()`):
  **Computes** the *sector-level* equal-weighted 12-month return (`indmom`) from prices.

* **The following block** (using `returns ... merge(indmom) ...`):
  **Does not compute** `indmom`; it **attaches/broadcasts** the already-computed sector value to each **(date, ticker)** row—so every stock in the same sector gets the same `indmom` on that date.

* **Broadcasting:** After the merge, `indmom` is **replicated across all tickers** that share the same `(date, sector)`. This is intentional if you want a *sector-level control* attached to each stock row.

* **Merge behavior:** `pd.DataFrame.merge` defaults to an **inner join**.
  If some `(date, sector)` combos exist in your returns panel but not in the `indmom` table (or vice versa), rows will be dropped.

  * To keep all stock-date rows, use:

    ```python
    .merge(indmom, how='left')
    ```

* **Alignment/frequency:** Ensure `returns` are aligned to the **same weekly grid** (e.g., `'W-FRI'`) used when computing `indmom`, otherwise the merge may drop rows due to date mismatches.

* **Join key for metadata:** `.join(metadata[['sector']])` relies on `metadata` being indexed by `ticker`. If it isn’t, use:

  ```python
  .join(metadata.set_index('ticker')[['sector']], on='ticker')
  ```

* **Block A (earlier):** “Build the sector feature.”
* **Block B (this one):** “Attach that sector feature to each stock-date row so it can be used in stock-level models.”

In [56]:
returns.head()

ticker,A,AA,AAL,AAMI,AAOI,AAON,AAP,AAPL,AAT,AB,...,ZM,ZS,ZTEK,ZTO,ZTR,ZTS,ZUMZ,ZVRA,ZWS,ZYME
date,,,,,,,,,,,,,,,,,,,,,
1990-01-12,NaN,-0.027915,NaN,NaN,NaN,NaN,NaN,-0.086093,NaN,-0.023437,...,NaN,NaN,NaN,NaN,-0.012659,NaN,NaN,NaN,NaN,NaN
1990-01-19,NaN,-0.111486,NaN,NaN,NaN,NaN,NaN,-0.007245,NaN,0.016001,...,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
1990-01-26,NaN,-0.049429,NaN,NaN,NaN,NaN,NaN,-0.043796,NaN,-0.023623,...,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN
1990-02-02,NaN,0.021416,NaN,NaN,NaN,NaN,NaN,0.045802,NaN,0.040323,...,NaN,NaN,NaN,NaN,-0.012820,NaN,NaN,NaN,NaN,NaN
1990-02-09,NaN,0.004033,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,-0.017258,...,NaN,NaN,NaN,NaN,0.008272,NaN,NaN,NaN,NaN,NaN


In [57]:
metadata.head()

,sector,sharesoutstanding,marketcap
ticker,,,
A,Healthcare,282602317.0,3.148005e+10
AA,Basic Materials,263839742.0,1.489375e+10
AACB,Financial Services,22175000.0,2.861595e+08
AACBU,Financial Services,25175000.0,2.693725e+08
AACG,Consumer Defensive,31772461.0,3.272563e+07


In [58]:
print(returns.stack())

date        ticker
1990-01-12  AA       -0.027915
            AAPL     -0.086093
            AB       -0.023437
            ABM       0.003344
            ABT      -0.022336
                        ...   
2020-01-03  ZTS      -0.006754
            ZUMZ      0.057239
            ZVRA     -0.093301
            ZWS      -0.002446
            ZYME      0.018141
Length: 2886291, dtype: float64


In [59]:
returns.stack().to_frame('ret').head()

ret
date       ticker          
1990-01-12 AA     -0.027915
           AAPL   -0.086093
           AB     -0.023437
           ABM     0.003344
           ABT    -0.022336

In [60]:
returns.stack().to_frame('ret').join(metadata[['sector']]).head()

ret              sector
date       ticker                              
1990-01-12 AA     -0.027915     Basic Materials
           AAPL   -0.086093          Technology
           AB     -0.023437  Financial Services
           ABM     0.003344         Industrials
           ABT    -0.022336          Healthcare

In [61]:
returns.stack().to_frame('ret').join(metadata[['sector']]).reset_index().head()

,date,ticker,ret,sector
0,1990-01-12,AA,-0.027915,Basic Materials
1,1990-01-12,AAPL,-0.086093,Technology
2,1990-01-12,AB,-0.023437,Financial Services
3,1990-01-12,ABM,0.003344,Industrials
4,1990-01-12,ABT,-0.022336,Healthcare


In [62]:
returns.stack().to_frame('ret').join(metadata[['sector']]).reset_index().merge(indmom).head()

,date,ticker,ret,sector,indmom
0,1991-01-04,AA,-0.013129,Basic Materials,-0.084250
1,1991-01-04,AAPL,0.005813,Technology,-0.108320
2,1991-01-04,AB,-0.038168,Financial Services,-0.124174
3,1991-01-04,ABM,-0.024155,Industrials,-0.134353
4,1991-01-04,ABT,-0.079096,Healthcare,0.107369


In [63]:
returns.stack().to_frame('ret').join(metadata[['sector']]).reset_index().merge(indmom).set_index(['date', 'ticker']).head()

ret              sector    indmom
date       ticker                                        
1991-01-04 AA     -0.013129     Basic Materials -0.084250
           AAPL    0.005813          Technology -0.108320
           AB     -0.038168  Financial Services -0.124174
           ABM    -0.024155         Industrials -0.134353
           ABT    -0.079096          Healthcare  0.107369

In [64]:
returns.stack().to_frame('ret').join(metadata[['sector']]).reset_index().merge(indmom).set_index(['date', 'ticker']).loc[:, ['indmom']].head()

indmom
date       ticker          
1991-01-04 AA     -0.084250
           AAPL   -0.108320
           AB     -0.124174
           ABM    -0.134353
           ABT     0.107369

In [65]:
indmom = (
    returns
    # returns: wide DataFrame (dates × tickers) of weekly returns
    .stack()
    # -> long Series with MultiIndex (date, ticker)
    .to_frame('ret')
    # -> DataFrame with a 'ret' column (not used later, but keeps structure)
    
    # Attach sector label per ticker.
    # Assumes 'metadata' is indexed by ticker (so join aligns on the 'ticker' level).
    .join(metadata[['sector']])
    
    # Make 'date' and 'ticker' ordinary columns for merging.
    .reset_index()
    
    # Merge with the PRECOMPUTED sector-level indmom table
    # (which has columns ['date', 'sector', 'indmom'] from the previous computation).
    # Default is inner merge on overlapping column names: here, 'date' and 'sector'.
    .merge(indmom)
    
    # Restore a panel index keyed by (date, ticker)
    .set_index(['date', 'ticker'])
    
    # Keep only the sector momentum column; it’s identical for all tickers in the same
    # sector at a given date (i.e., sector value broadcast to each member stock).
    .loc[:, ['indmom']]
)


In [66]:
indmom.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2847760 entries, (Timestamp('1991-01-04 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   indmom  float64
dtypes: float64(1)
memory usage: 32.8+ MB


In [67]:
indmom.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/indmom')
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

#### Recent Max Return

**Max daily returns from calendar month t-1**

For each **stock** and each **Friday**, this variable gives:

* The **maximum 21-day return** (computed as `pct_change(periods=MONTH)`) observed during the **previous month (≈21 trading days)**.

So for week $t$ and stock $i$,

$$
\text{maxret}_{i,t} = \max \left\{ \frac{P_{i,k}}{P_{i,k-21}} - 1 \;\;:\;\; k \in [t-20, \dots, t] \right\}
$$

* **Why maximum daily (or monthly) return?**
  Researchers sometimes use “max daily return” as a **lottery-like characteristic**:

  * Some stocks occasionally experience extreme positive jumps (like penny stocks or distressed firms).
  * Investors who chase these “lottery tickets” often overpay, which predicts **lower future returns** (Kumar 2009, Bali et al. 2011).

* **Here:** by computing the **max 21-day return in the past month**, you’re capturing whether the stock recently experienced an unusually large upside move.

  * High `maxret` = stock behaved like a lottery (large spike).
  * Empirically, such stocks tend to **underperform subsequently** because investors are drawn to them despite poor fundamentals.
---

**Important note on the code:**
Using `.pct_change(periods=MONTH)` (≈21 days) means we are actually measuring the **maximum 21-day return in the past month**.
If we truly wanted the **maximum daily return in the past month**, the line should be:

```python
close.pct_change(periods=1).rolling(21).max()
```
---

In [68]:
# Compute Max Daily Return over the prior calendar month (t-1)
# ------------------------------------------------------------

maxret = (
    close
    # 1) Compute 1-day percentage changes in closing prices.
    #    NOTE: periods=MONTH here looks odd because MONTH≈21 means 21 trading days (≈1 month).
    #    If the intention is "daily returns", one would normally use periods=1.
    #    With periods=MONTH, you're computing returns over ~21 days. 
    #    (So this code is actually measuring *21-day returns*, not single-day returns.)
    .pct_change(periods=MONTH)
    # 2) For each date and ticker, compute the maximum return within the past 21 trading days.
    #    .rolling(21).max() looks at the rolling window of length 21 (≈1 month).
    #    So this keeps the "largest 21-day return observed in the last month".
    .rolling(21).max()
    # 3) Resample to weekly frequency, taking the last available value on each Friday.
    #    This ensures signals are aligned to a weekly panel and reduces daily noise.
    .resample('W-FRI').last()
    # 4) Convert from wide format (dates × tickers) to long format (MultiIndex of date × ticker).
    .stack()
    # 5) Wrap the result in a DataFrame with a descriptive column name.
    .to_frame('maxret')
)

In [69]:
maxret.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2862201 entries, (Timestamp('1990-03-02 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   maxret  float64
dtypes: float64(1)
memory usage: 32.9+ MB


In [70]:
maxret.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/maxret')
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

#### Long-Term Reversal

Cumulative returns months t-36 to t-13.

In [71]:
mom36m = (close
           .pct_change(periods=24*MONTH)
           .shift(12*MONTH)
           .resample('W-FRI')
           .last()
           .stack()
           .to_frame('mom36m'))

In [72]:
mom36m.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2417903 entries, (Timestamp('1993-01-01 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZWS')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   mom36m  float64
dtypes: float64(1)
memory usage: 27.9+ MB


In [73]:
mom36m.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/mom36m')
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231]) 
/factor/beta                frame        (shape->[2419319,1]) 
/factor/betasq              frame        (shape->[2419319,1]) 
/factor/chmom               frame        (shape->[2724572,1]) 
/factor/dolvol              frame        (shape->[2862201,1]) 
/factor/idiovol             frame        (shape->[2419319,1]) 
/factor/ill                 frame        (shape->[2584218,1]) 
/factor/indmom              frame        (shape->[2847760,1]) 
/factor/maxret              frame        (shape->[2862201,1]) 
/factor/mom12m              frame        (shape->[2724572,1]) 
/factor/mom1m               series       (shape->[2875516])   
/factor/mom36m              frame        (shape->[2417903,1]) 
/factor/mvel                frame        (shape->[2889522,1]) 
/factor/retvol              frame        (shape->[2875516,1]) 
/factor/turn   

### Liquidity Metrics

#### Turnover

Avg. monthly trading volume for most recent three months scaled by number of shares; we are using the most recent no of shares from yahoo finance

In [74]:
turn = (volume
        .rolling(3*MONTH)
        .mean()
        .resample('W-FRI')
        .last()
        .div(metadata.sharesoutstanding)
        .stack('ticker')
        .to_frame('turn'))

In [75]:
turn.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2848548 entries, (Timestamp('1990-03-30 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   turn    float64
dtypes: float64(1)
memory usage: 32.8+ MB


In [73]:
turn.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/turn')

In [74]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                    frame        (shape->[7559,3231])
/factor/chmom             frame        (shape->[2724572,1])
/factor/indmom            frame        (shape->[2847760,1])
/factor/maxret            frame        (shape->[2862201,1])
/factor/mom12m            frame        (shape->[2724572,1])
/factor/mom1m             series       (shape->[2875516])  
/factor/mom36m            frame        (shape->[2417903,1])
/factor/turn              frame        (shape->[2848548,1])
/metadata                 frame        (shape->[1,3])      
/returns                  frame        (shape->[1565,3231])
/volume                   frame        (shape->[7559,3231])


#### Turnover Volatility

Monthly std dev of daily share turnover

In [75]:
turn_std = (prices
            .volume
            .unstack('ticker')
            .div(metadata.sharesoutstanding)
            .rolling(MONTH)
            .std()
            .resample('W-FRI')
            .last()
            .stack('ticker')
            .to_frame('turn_std'))

In [76]:
turn_std.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/turn_std')

In [77]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/chmom               frame        (shape->[2724572,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            frame        (shape->[2874701,1])
/metadata                   frame        (shape->[1,3])      
/returns                    frame        (shape->[1565,3231])
/volume                     frame        (shape->[7559,3231])


#### Log Market Equity

Natural log of market cap at end of month t-1

In [78]:
last_price = close.ffill()
factor = close.div(last_price.iloc[-1])
mvel = np.log1p(factor.mul(metadata.marketcap).resample('W-FRI').last()).stack().to_frame('mvel')

In [79]:
mvel.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2889522 entries, (Timestamp('1990-01-05 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   mvel    float64
dtypes: float64(1)
memory usage: 33.3+ MB


In [80]:
mvel.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/mvel')

In [81]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/chmom               frame        (shape->[2724572,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            frame        (shape->[2874701,1])
/metadata                   frame        (shape->[1,3])      
/returns                    frame        (shape->[1565,3231])
/volume                     frame        (shape->[7559,3231])


#### Dollar Volume

Natural log of trading volume time price per share from month t-2

In [82]:
dv = close.mul(volume)

In [83]:
dolvol = (np.log1p(dv.rolling(21)
                  .mean()
                  .shift(21)
                  .resample('W-FRI')
                  .last())
          .stack()
          .to_frame('dolvol'))

In [84]:
dolvol.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/dolvol')

In [85]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/chmom               frame        (shape->[2724572,1])
/factor/dolvol              frame        (shape->[2862201,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            frame        (shape->[2874701,1])
/metadata                   frame        (shape->[1,3])      
/returns                    frame        (shape->[1565,3231])
/volume                     frame        (shape->[7559,3231])


#### Amihud Illiquidity

Average of daily (absolute return / dollar volume)

In [86]:
ill = (close.pct_change().abs()
       .div(dv)
       .rolling(21)
       .mean()
       .resample('W-FRI').last()
       .stack()
       .to_frame('ill'))

In [87]:
ill.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2584218 entries, (Timestamp('1990-02-02 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   ill     float64
dtypes: float64(1)
memory usage: 29.8+ MB


In [90]:
ill.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/ill')

In [91]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/chmom               frame        (shape->[2724572,1])
/factor/dolvol              frame        (shape->[2862201,1])
/factor/ill                 frame        (shape->[2584218,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            frame        (shape->[2874701,1])
/metadata                   frame        (shape->[1,3])      
/returns                    frame        (shape->[1565,3231])
/volume                     f

### Risk Measures

#### Return Volatility

Standard dev of daily returns from month t-1.

In [92]:
retvol = (close.pct_change()
          .rolling(21)
          .std()
          .resample('W-FRI')
          .last()
          .stack()
          .to_frame('retvol'))

In [93]:
retvol.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2875516 entries, (Timestamp('1990-02-02 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZYME')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   retvol  float64
dtypes: float64(1)
memory usage: 33.1+ MB


In [94]:
retvol.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/retvol')

In [95]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/chmom               frame        (shape->[2724572,1])
/factor/dolvol              frame        (shape->[2862201,1])
/factor/ill                 frame        (shape->[2584218,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/retvol              frame        (shape->[2875516,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            frame        (shape->[2874701,1])
/metadata                   frame        (shape->[1,3])      
/returns                    f

#### Market Beta

Estimated market beta from weekly returns and equal weighted market returns for 3 years ending month t-1 with at least 52 weeks of returns.

In [96]:
index = close.resample('W-FRI').last().pct_change().mean(1).to_frame('x')

**Short Description**

1. **`close.resample('W-FRI').last()`**

   * Convert daily closes to **weekly (Friday) closes** for every ticker.

2. **`.pct_change()`**

   * Convert weekly closes to **weekly returns** for each ticker (column-wise).

3. **`.mean(1)`**

   * Take the **cross-sectional mean across tickers** on each week (row-wise).
   * This is an **equal-weighted market return** (no cap weights, skip-na by default).

4. **`.to_frame('x')`**

   * Store the market return in a DataFrame with a single column named **`x`**.

**Long Description**

After this line:

```python
weekly_returns = close.resample('W-FRI').last().pct_change()
```

you have a **matrix** (a pandas DataFrame):

* **Rows (axis=0):** dates (each Friday).
* **Columns (axis=1):** tickers.
* **Entries:** $r_{i,t}$, the weekly return of stock $i$ in week $t$.

So if you have 3,000 tickers and 1,000 Fridays, the shape is **(1000 × 3000)**.


**What `.mean(1)` does**

`.mean(1)` = take the mean **across columns** for each row.

* For a given week $t$, you don’t look at the *time series* yet — you look across all **stocks** in that week.
* That’s why it’s called the **cross-sectional mean**.

Formally:

$$
x_t = \frac{1}{N_t} \sum_{i=1}^{N_t} r_{i,t}
$$

Here $N_t$ is the number of tickers that reported a return that week.

**Why this is a market proxy**

You want an **equal-weighted “market index” return** each week.

* At 2020-01-03, if 3000 stocks traded, you average their returns → $N_t = 3000$.
* At 2020-01-10, if only 2995 stocks had valid data, $N_t = 2995$.

So the result of `.mean(1)` is a **single time series** (length = number of Fridays) where each entry is the **average return across all stocks that week**.

In [97]:
def get_market_beta(y, x=index):
    df = x.join(y.to_frame('y')).dropna()
    model = RollingOLS(endog=df.y, 
                       exog=sm.add_constant(df[['x']]),
                      window=3*52)

    return model.fit(params_only=True).params['x']

5. **Inputs**

   * `y`: a **Series** of weekly returns for **one ticker** (one column of your `returns` panel).
   * `x`: the **equal-weighted market return** you built above (defaults to `index`).

<p>
    
6. **`df = x.join(y.to_frame('y')).dropna()`**

   * Align market and stock returns by date; **drop weeks with missing values**.
   * Result has two columns: `x` (market) and `y` (stock).

<p>
    
7. **`RollingOLS(..., window=3*52)`**

   * Run a **rolling OLS regression** with a **right-aligned** 156-week window:

     $$
     y_t = \alpha_t + \beta_t\, x_t + \varepsilon_t \quad
     \text{estimated over } \{t-155,\dots,t\}.
     $$
   * `sm.add_constant(...)` includes an intercept $\alpha_t$.

<p>
    
8. **`fit(params_only=True).params['x']`**

   * Fit the rolling model and return only the **slope** series $\beta_t$ (the coefficient on `x`).
   * You get a **time series of betas**, indexed by week; the first \~155 weeks are `NaN` until the window fills.

<p>
    
9. **`returns.dropna(thresh=3*52, axis=1)`**

   * Keep only tickers with **at least 156 non-missing weekly returns** in the entire sample (stricter than “52 weeks” mentioned in the prose).

<p>
    
10. **`.apply(get_market_beta)`**

* Apply the function **column-wise**: for each ticker’s return series `y`, compute its rolling **beta series**.

<p>
    
11. **`.stack().to_frame('beta')`**

* Convert the wide matrix of betas (weeks × tickers) to **long format** with index `(date, ticker)` and a single column **`beta`**.


**Analytical view**

For each stock $i$ and week $t$, the reported beta is the OLS slope over the last $W=156$ weeks:

$$
\hat\beta_{i,t} \;=\; 
\frac{\sum_{k=t-W+1}^t (x_k-\bar{x})(y_{i,k}-\bar{y}_i)}
     {\sum_{k=t-W+1}^t (x_k-\bar{x})^2},
$$

where $x_k$ is the **equal-weighted market return** in week $k$, and $y_{i,k}$ is the stock’s weekly return. (Because an intercept is included, the OLS slope equals the covariance/variance ratio within the rolling window.)

* The estimate at date $t$ is **right-aligned**: it uses data up to week $t$ (≈ “ending week $t$”).
* If you later want a **“beta as of month $t-1$”**, you typically align to the **last Friday in month $t-1$** (e.g., by grouping to month-end and taking the last available beta).



In [98]:
beta = (returns.dropna(thresh=3*52, axis=1)
        .apply(get_market_beta).stack().to_frame('beta'))

In [99]:
beta.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2419319 entries, (Timestamp('1993-01-01 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZWS')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   beta    float64
dtypes: float64(1)
memory usage: 27.8+ MB


In [100]:
beta.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/beta')

In [101]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/beta                frame        (shape->[2419319,1])
/factor/chmom               frame        (shape->[2724572,1])
/factor/dolvol              frame        (shape->[2862201,1])
/factor/ill                 frame        (shape->[2584218,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/retvol              frame        (shape->[2875516,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            frame        (shape->[2874701,1])
/metadata                   f

#### Beta Squared

Market beta squared

In [102]:
betasq = beta.beta.pow(2).to_frame('betasq')

In [103]:
betasq.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2419319 entries, (Timestamp('1993-01-01 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZWS')
Data columns (total 1 columns):
 #   Column  Dtype  
---  ------  -----  
 0   betasq  float64
dtypes: float64(1)
memory usage: 27.8+ MB


In [104]:
betasq.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/betasq')

In [105]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/beta                frame        (shape->[2419319,1])
/factor/betasq              frame        (shape->[2419319,1])
/factor/chmom               frame        (shape->[2724572,1])
/factor/dolvol              frame        (shape->[2862201,1])
/factor/ill                 frame        (shape->[2584218,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/retvol              frame        (shape->[2875516,1])
/factor/turn                frame        (shape->[2848548,1])
/factor/turn_std            f

#### Idiosyncratic return volatility

Standard dev of a regression of residuals of weekly returns on the returns of an equal weighted market index returns for the prior three years.

This takes a while!

In [106]:
def get_ols_residuals(y, x=index):
    df = x.join(y.to_frame('y')).dropna()
    model = sm.OLS(endog=df.y, exog=sm.add_constant(df[['x']]))
    result = model.fit()
    return result.resid.std()

In [107]:
idiovol = (returns.apply(lambda x: x.rolling(3 * 52)
                         .apply(get_ols_residuals)))

In [108]:
idiovol = idiovol.stack().to_frame('idiovol')

In [109]:
idiovol.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 2419319 entries, (Timestamp('1993-01-01 00:00:00'), 'AA') to (Timestamp('2020-01-03 00:00:00'), 'ZWS')
Data columns (total 1 columns):
 #   Column   Dtype  
---  ------   -----  
 0   idiovol  float64
dtypes: float64(1)
memory usage: 27.8+ MB


In [110]:
idiovol.to_hdf(results_path / 'autoencoder_reloaded.h5', 'factor/idiovol')

In [111]:
with pd.HDFStore(results_path / 'autoencoder_reloaded.h5') as store:
    print(store.info())

<class 'pandas.io.pytables.HDFStore'>
File path: c:\data\ip_2026\asset_pricing\autoencoder_reloaded.h5
/close                      frame        (shape->[7559,3231])
/factor/beta                frame        (shape->[2419319,1])
/factor/betasq              frame        (shape->[2419319,1])
/factor/chmom               frame        (shape->[2724572,1])
/factor/dolvol              frame        (shape->[2862201,1])
/factor/idiovol             frame        (shape->[2419319,1])
/factor/ill                 frame        (shape->[2584218,1])
/factor/indmom              frame        (shape->[2847760,1])
/factor/maxret              frame        (shape->[2862201,1])
/factor/mom12m              frame        (shape->[2724572,1])
/factor/mom1m               series       (shape->[2875516])  
/factor/mom36m              frame        (shape->[2417903,1])
/factor/mvel                frame        (shape->[2889522,1])
/factor/retvol              frame        (shape->[2875516,1])
/factor/turn                f